# Chain-of-Thought Reasoning for Narrative Similarity


In [1]:
import json
import google.generativeai as genai
import pandas as pd
import time
from typing import Dict, List, Tuple
import re

In [3]:
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel('gemini-2.5-flash')

print("Gemini API configured successfully!")

Gemini API configured successfully!


## Step 1: Enhanced Extraction with Reasoning



In [4]:
EXTRACTION_PROMPT_COT = """Analyze this story and extract symbolic narrative elements.

Story: {story}

Think step-by-step and extract:

1. **Themes**: What are the core ideas or messages? (2-4 keywords)
   - Why did you identify these themes?

2. **Key Events**: What are the main plot points in chronological order? (4-6 events)
   - Focus on turning points and major actions

3. **Outcomes**: How does the story resolve? (1-2 outcomes)
   - What changes by the end?

4. **Conflict Type**: What is the central conflict?
   - Person vs. Person, Person vs. Self, Person vs. Society, Person vs. Nature, etc.

Return ONLY valid JSON:
{{
    "themes": ["theme1", "theme2"],
    "theme_reasoning": "brief explanation",
    "events": ["event1", "event2", "event3"],
    "outcomes": ["outcome1"],
    "conflict_type": "conflict"
}}
"""

def extract_elements_with_reasoning(story: str, max_retries: int = 3) -> Dict:
    """
    Extract narrative elements from a story with reasoning.
    """
    for attempt in range(max_retries):
        try:
            prompt = EXTRACTION_PROMPT_COT.format(story=story)
            response = model.generate_content(prompt)
            
            text = response.text.strip()
            if "```json" in text:
                text = text.split("```json")[1].split("```")[0]
            elif "```" in text:
                text = text.split("```")[1].split("```")[0]
            text = text.strip()
            
            elements = json.loads(text)
            
            required = ["themes", "events", "outcomes"]
            if all(key in elements for key in required):
                return elements
            else:
                print(f"Missing required fields on attempt {attempt + 1}")
                
        except json.JSONDecodeError as e:
            print(f"JSON error on attempt {attempt + 1}: {e}")
            time.sleep(1)
        except Exception as e:
            print(f"Error on attempt {attempt + 1}: {e}")
            time.sleep(1)
    
    print("Warning: All extraction attempts failed, using empty structure")
    return {
        "themes": [],
        "theme_reasoning": "",
        "events": [],
        "outcomes": [],
        "conflict_type": ""
    }

## Step 2: Chain-of-Thought Comparison


In [5]:
COT_COMPARISON_PROMPT = """You are an expert in narrative analysis. Compare these stories using step-by-step reasoning.

ANCHOR STORY:
Themes: {anchor_themes}
Events: {anchor_events}
Outcomes: {anchor_outcomes}
Conflict: {anchor_conflict}

OPTION A:
Themes: {a_themes}
Events: {a_events}
Outcomes: {a_outcomes}
Conflict: {a_conflict}

OPTION B:
Themes: {b_themes}
Events: {b_events}
Outcomes: {b_outcomes}
Conflict: {b_conflict}

---

STEP-BY-STEP ANALYSIS:

Step 1: Compare Themes
- How many themes overlap between Anchor and Option A? List them.
- How many themes overlap between Anchor and Option B? List them.
- Score: Give Option A a score from 0-10, and Option B a score from 0-10.

Step 2: Compare Event Sequences
- Do Option A's events follow a similar pattern to Anchor's events?
- Do Option B's events follow a similar pattern to Anchor's events?
- Consider: similar action types, similar progressions, similar turning points
- Score: Give Option A a score from 0-10, and Option B a score from 0-10.

Step 3: Compare Outcomes
- How similar are Option A's outcomes to Anchor's outcomes?
- How similar are Option B's outcomes to Anchor's outcomes?
- Score: Give Option A a score from 0-10, and Option B a score from 0-10.

Step 4: Compare Conflict Types
- Does Option A have the same type of conflict as Anchor?
- Does Option B have the same type of conflict as Anchor?
- Score: Give Option A a score from 0-10, and Option B a score from 0-10.

Step 5: Final Decision
- Total Score A: (sum of all A scores)
- Total Score B: (sum of all B scores)
- Decision: Which option has higher total score?

Return ONLY valid JSON:
{{
    "step1_themes": {{
        "a_overlaps": ["theme1", "theme2"],
        "b_overlaps": ["theme1"],
        "score_a": 8,
        "score_b": 5,
        "reasoning": "brief explanation"
    }},
    "step2_events": {{
        "score_a": 7,
        "score_b": 6,
        "reasoning": "brief explanation"
    }},
    "step3_outcomes": {{
        "score_a": 6,
        "score_b": 8,
        "reasoning": "brief explanation"
    }},
    "step4_conflict": {{
        "score_a": 9,
        "score_b": 4,
        "reasoning": "brief explanation"
    }},
    "step5_final": {{
        "total_a": 30,
        "total_b": 23,
        "decision": "A",
        "reasoning": "Option A has higher total score"
    }}
}}
"""

def chain_of_thought_comparison(
    anchor_elements: Dict,
    a_elements: Dict,
    b_elements: Dict,
    max_retries: int = 3
) -> Tuple[bool, Dict]:
    """
    Compare stories using chain-of-thought reasoning.
    Returns: (text_a_is_closer, reasoning_chain)
    """
    prompt = COT_COMPARISON_PROMPT.format(
        anchor_themes=", ".join(anchor_elements.get("themes", [])),
        anchor_events=" → ".join(anchor_elements.get("events", [])),
        anchor_outcomes=", ".join(anchor_elements.get("outcomes", [])),
        anchor_conflict=anchor_elements.get("conflict_type", ""),
        
        a_themes=", ".join(a_elements.get("themes", [])),
        a_events=" → ".join(a_elements.get("events", [])),
        a_outcomes=", ".join(a_elements.get("outcomes", [])),
        a_conflict=a_elements.get("conflict_type", ""),
        
        b_themes=", ".join(b_elements.get("themes", [])),
        b_events=" → ".join(b_elements.get("events", [])),
        b_outcomes=", ".join(b_elements.get("outcomes", [])),
        b_conflict=b_elements.get("conflict_type", "")
    )
    
    for attempt in range(max_retries):
        try:
            response = model.generate_content(prompt)
            text = response.text.strip()
            
            if "```json" in text:
                text = text.split("```json")[1].split("```")[0]
            elif "```" in text:
                text = text.split("```")[1].split("```")[0]
            text = text.strip()
            
            # Parse the reasoning chain
            reasoning_chain = json.loads(text)
            
            # Extract the decision
            decision = reasoning_chain.get("step5_final", {}).get("decision", "A")
            decision = decision.strip().upper()
            
            # Return True if A is closer, False if B is closer
            if 'A' in decision and 'B' not in decision:
                return True, reasoning_chain
            elif 'B' in decision and 'A' not in decision:
                return False, reasoning_chain
            elif decision.startswith('A'):
                return True, reasoning_chain
            else:
                return False, reasoning_chain
                
        except json.JSONDecodeError as e:
            print(f"JSON error on attempt {attempt + 1}: {e}")
            time.sleep(1)
        except Exception as e:
            print(f"Error on attempt {attempt + 1}: {e}")
            time.sleep(1)
    
    print("Warning: All comparison attempts failed, defaulting to A")
    return True, {}

## Step 3: Main Prediction Function



In [6]:
def predict_with_cot(
    anchor: str,
    text_a: str,
    text_b: str,
    verbose: bool = False
) -> Tuple[bool, Dict]:
    """
    Main prediction function using Chain-of-Thought reasoning.
    
    Args:
        anchor: The anchor story text
        text_a: Option A story text
        text_b: Option B story text
        verbose: If True, print intermediate steps
    
    Returns:
        (text_a_is_closer, full_reasoning)
    """
    if verbose:
        print("\n" + "="*60)
        print("Extracting narrative elements...")
        print("="*60)
    
    # Step 1: Extract elements from all three stories
    anchor_elements = extract_elements_with_reasoning(anchor)
    a_elements = extract_elements_with_reasoning(text_a)
    b_elements = extract_elements_with_reasoning(text_b)
    
    if verbose:
        print("\nAnchor themes:", anchor_elements.get("themes", []))
        print("Option A themes:", a_elements.get("themes", []))
        print("Option B themes:", b_elements.get("themes", []))
        print("\n" + "="*60)
        print("Performing Chain-of-Thought comparison...")
        print("="*60)
    
    # Step 2: Compare with chain-of-thought reasoning
    text_a_is_closer, reasoning_chain = chain_of_thought_comparison(
        anchor_elements,
        a_elements,
        b_elements
    )
    
    if verbose and reasoning_chain:
        print("\nReasoning Chain:")
        if "step5_final" in reasoning_chain:
            final = reasoning_chain["step5_final"]
            print(f"  Total Score A: {final.get('total_a', 'N/A')}")
            print(f"  Total Score B: {final.get('total_b', 'N/A')}")
            print(f"  Decision: {final.get('decision', 'N/A')}")
            print(f"  Reasoning: {final.get('reasoning', 'N/A')}")
    
    # Compile full reasoning for debugging/analysis
    full_reasoning = {
        "anchor_elements": anchor_elements,
        "a_elements": a_elements,
        "b_elements": b_elements,
        "reasoning_chain": reasoning_chain,
        "prediction": "A" if text_a_is_closer else "B"
    }
    
    return text_a_is_closer, full_reasoning

## Step 4: Test on a Single Example



In [7]:
dev_df = pd.read_json('../Data/SemEval2026-Task_4-dev-v1/dev_track_a.jsonl', lines=True)
print(f"Loaded {len(dev_df)} development examples")
print(f"\nColumns: {dev_df.columns.tolist()}")

Loaded 200 development examples

Columns: ['anchor_text', 'text_a', 'text_b', 'text_a_is_closer']


In [8]:
# Test on the first example
example = dev_df.iloc[0]

print("Testing Chain-of-Thought reasoning on first example...\n")
print(f"Ground truth: {'A' if example['text_a_is_closer'] else 'B'}")
print()

prediction, reasoning = predict_with_cot(
    example['anchor_text'],
    example['text_a'],
    example['text_b'],
    verbose=True
)

print(f"\n{'='*60}")
print(f"RESULT: Predicted {'A' if prediction else 'B'}")
print(f"Correct: {prediction == example['text_a_is_closer']}")
print(f"{'='*60}")

Testing Chain-of-Thought reasoning on first example...

Ground truth: B


Extracting narrative elements...

Anchor themes: ['Climate Action', 'Intergenerational Justice', 'Economic Transformation', 'Technological Innovation']
Option A themes: ['Determination', 'Justice', 'Unforeseen Heroism', 'Sentimental Value']
Option B themes: ['Survival', 'Resource Scarcity', 'Conflict', 'Discovery']

Performing Chain-of-Thought comparison...

Reasoning Chain:
  Total Score A: 12
  Total Score B: 23
  Decision: B
  Reasoning: Option B has a significantly higher total score due to stronger thematic overlaps (survival as a core goal, technological innovation, presence of conflict), a more aligned scale and impact in its event sequence and outcomes (global crisis, species-level implications), despite having a different type of core conflict.

RESULT: Predicted B
Correct: True


## Step 5: Evaluate on Full Development Set



In [11]:
def evaluate_cot(df: pd.DataFrame, max_samples: int = None) -> Tuple[float, pd.DataFrame]:
    """
    Evaluate Chain-of-Thought approach on a dataset.
    
    Args:
        df: DataFrame with columns ['anchor_text', 'text_a', 'text_b', 'text_a_is_closer']
        max_samples: If set, only evaluate on first N samples (for testing)
    
    Returns:
        (accuracy, results_df)
    """
    if max_samples:
        df = df.head(max_samples)
    
    predictions = []
    correct = 0
    total = len(df)
    
    print(f"Evaluating on {total} examples...")
    
    for idx, row in df.iterrows():
        if (idx + 1) % 10 == 0:
            print(f"Progress: {idx + 1}/{total} ({(idx + 1)/total*100:.1f}%) - Current accuracy: {correct/(idx+1)*100:.2f}%")
        
        try:
            prediction, reasoning = predict_with_cot(
                row['anchor_text'],
                row['text_a'],
                row['text_b'],
                verbose=False
            )
            
            # Check if correct
            is_correct = (prediction == row['text_a_is_closer'])
            if is_correct:
                correct += 1
            
            predictions.append({
                'idx': idx,
                'prediction': prediction,
                'ground_truth': row['text_a_is_closer'],
                'correct': is_correct,
                'reasoning': reasoning
            })
            
        except Exception as e:
            print(f"\nError on example {idx}: {e}")
            # Record as incorrect if we fail
            predictions.append({
                'idx': idx,
                'prediction': True,  # Default to A
                'ground_truth': row['text_a_is_closer'],
                'correct': False,
                'reasoning': {}
            })
    
    accuracy = correct / total
    results_df = pd.DataFrame(predictions)
    
    print(f"\n{'='*60}")
    print(f"EVALUATION COMPLETE")
    print(f"{'='*60}")
    print(f"Total examples: {total}")
    print(f"Correct: {correct}")
    print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"{'='*60}")
    
    return accuracy, results_df

In [12]:
print("Testing on first 10 examples...\n")
test_accuracy, test_results = evaluate_cot(dev_df, max_samples=10)

print("\nFirst 5 predictions:")
print(test_results[['idx', 'prediction', 'ground_truth', 'correct']].head())

Testing on first 10 examples...

Evaluating on 10 examples...
Progress: 10/10 (100.0%) - Current accuracy: 40.00%

EVALUATION COMPLETE
Total examples: 10
Correct: 4
Accuracy: 0.4000 (40.00%)

First 5 predictions:
   idx  prediction  ground_truth  correct
0    0       False         False     True
1    1       False          True    False
2    2       False         False     True
3    3        True         False    False
4    4       False         False     True


In [13]:
print(test_results[['idx', 'prediction', 'ground_truth', 'correct']])

   idx  prediction  ground_truth  correct
0    0       False         False     True
1    1       False          True    False
2    2       False         False     True
3    3        True         False    False
4    4       False         False     True
5    5       False         False     True
6    6        True         False    False
7    7        True         False    False
8    8       False          True    False
9    9       False          True    False


In [14]:
example = dev_df.iloc[1]  
prediction, reasoning = predict_with_cot(
    example['anchor_text'],
    example['text_a'],
    example['text_b'],
    verbose=True
)

if 'reasoning_chain' in reasoning:
    chain = reasoning['reasoning_chain']
    print("\nDetailed Scores:")
    print(f"Step 1 (Themes): A={chain.get('step1_themes', {}).get('score_a')}, B={chain.get('step1_themes', {}).get('score_b')}")
    print(f"Step 2 (Events): A={chain.get('step2_events', {}).get('score_a')}, B={chain.get('step2_events', {}).get('score_b')}")
    print(f"Step 3 (Outcomes): A={chain.get('step3_outcomes', {}).get('score_a')}, B={chain.get('step3_outcomes', {}).get('score_b')}")
    print(f"Step 4 (Conflict): A={chain.get('step4_conflict', {}).get('score_a')}, B={chain.get('step4_conflict', {}).get('score_b')}")
    print(f"\nFinal: Total A={chain.get('step5_final', {}).get('total_a')}, Total B={chain.get('step5_final', {}).get('total_b')}")


Extracting narrative elements...

Anchor themes: ['Redemption', 'Prejudice', 'Growth']
Option A themes: ['Moral Dilemma', 'Familial Loyalty', 'Consequences of Trauma', 'Shifting Morality']
Option B themes: ['Identity', 'Manipulation', 'Conformity vs. Freedom']

Performing Chain-of-Thought comparison...

Reasoning Chain:
  Total Score A: 7
  Total Score B: 31
  Decision: B
  Reasoning: Option B has a significantly higher total score due to stronger thematic, event sequence, and outcome similarities, along with an identical conflict type.

Detailed Scores:
Step 1 (Themes): A=3, B=7
Step 2 (Events): A=2, B=6
Step 3 (Outcomes): A=2, B=8
Step 4 (Conflict): A=0, B=10

Final: Total A=7, Total B=31


In [ ]:
## Since COT shows a very large bias towards Option B, we wont be testing on the entire dev set